# Smoke-эксперимент: классификация товарных категорий

## tl;dr

На детерминированном synthetic smoke-наборе seller-group test содержит 9 строк от трёх ранее не встречавшихся продавцов. Фактический прогон ниже даёт macro-F1 = 1.000 и top-2 accuracy = 1.000. Это проверка исполнимости, а не оценка качества на реальном каталоге.

## Context & Methods

Notebook импортирует production-код из `src/`, читает committed `data/smoke.csv`, вызывает реальные функции `train()` и `evaluate()` и сохраняет временные артефакты вне репозитория. Модель объединяет word- и char-level TF-IDF.

### Key Assumptions

- seed зафиксирован равным 42;
- один `seller_id` целиком относится только к одному split;
- синтетические строки не используются для содержательных выводов;
- сетевые загрузки отсутствуют.

In [1]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

SEED = 42
PROJECT_ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / 'pyproject.toml').exists()
)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from product_category_classification.data import load_csv, split_by_seller  # noqa: E402
from product_category_classification.evaluate import evaluate  # noqa: E402
from product_category_classification.predict import predict  # noqa: E402
from product_category_classification.train import train as train_model  # noqa: E402

DATA_PATH = PROJECT_ROOT / 'data' / 'smoke.csv'

## Data

Набор полностью синтетический и хранится в репозитории. Проверяем схему и отсутствие seller leakage до обучения.

In [2]:
products = load_csv(DATA_PATH)
train_frame, validation_frame, test_frame, split_manifest = split_by_seller(
    products, random_state=SEED
)
seller_sets = [set(part['seller_id']) for part in (train_frame, validation_frame, test_frame)]
assert seller_sets[0].isdisjoint(seller_sets[1])
assert seller_sets[0].isdisjoint(seller_sets[2])
assert seller_sets[1].isdisjoint(seller_sets[2])

pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'rows': [len(train_frame), len(validation_frame), len(test_frame)],
    'sellers': [
        part['seller_id'].nunique()
        for part in (train_frame, validation_frame, test_frame)
    ],
    'categories': [
        part['category'].nunique()
        for part in (train_frame, validation_frame, test_frame)
    ],
})

,split,rows,sellers,categories
0,train,18,6,3
1,validation,9,3,3
2,test,9,3,3


## Results

Обучаем реальный pipeline, затем оцениваем только сохранённый test artifact.

In [3]:
temporary_directory = TemporaryDirectory(prefix='product-category-smoke-')
artifact_dir = Path(temporary_directory.name) / 'artifacts'
training_metadata = train_model(DATA_PATH, artifact_dir, random_state=SEED)
test_metrics = evaluate(artifact_dir / 'model.joblib', artifact_dir / 'test.csv')

pd.Series({
    'test_rows': test_metrics['rows'],
    'macro_f1': test_metrics['macro_f1'],
    'top_2_accuracy': test_metrics['top_2_accuracy'],
}, name='synthetic smoke')

test_rows         9.0
macro_f1          1.0
top_2_accuracy    1.0
Name: synthetic smoke, dtype: float64

In [4]:
sample_prediction = predict(
    artifact_dir / 'model.joblib', 'wireless gaming mouse', top_k=2
)
sample_prediction

{'prediction': 'electronics',
 'candidates': [{'category': 'electronics', 'probability': 0.5988710267155141},
  {'category': 'sports', 'probability': 0.2079917036725202}]}

## Takeaways

- Фактический test split из 9 строк не пересекается с train/validation по продавцам.
- Macro-F1 и top-2 accuracy равны 1.000 только на простом synthetic smoke-наборе.
- Следующий содержательный шаг — оценка на UCI PriceRunner с тем же seller-group протоколом и анализом редких категорий.